# Module 6.3: Preference Alignment (DPO)

Welcome to the ultimate stage of modern foundation models!

In standard SFT (the previous notebook), we give the model a Prompt and a Perfect Answer. The model learns to replicate it point-blank.
However, human preference is messy. An answer like *"Sure, here is the code: ..."* is much more preferred over an answer like *"I can help you with that. The code you need is: ..."* because humans love brevity. How do we mathematically train for subjective "vibes" and safety?

## 1. DPO: A Simpler Alternative to RLHF

The original recipe for aligning models is **RLHF (Reinforcement Learning from Human Feedback)**, usually with an algorithm called `PPO`.
It requires keeping several models around during training:
1. The frozen base/reference model (to keep the policy from drifting too far).
2. A separate reward model (trained to grade answers).
3. The active model being trained.
4. (Plus a value/critic model in standard PPO.)

That is a lot of moving parts and memory. RLHF still works and is still widely used in practice, but it is fiddly to get right.

**DPO (Direct Preference Optimization)** is a simpler alternative. It reaches a similar goal, matching human preferences, without training a separate reward model or running reinforcement learning. As we'll see, it turns preference data directly into an ordinary supervised-style loss.

## 2. Enter DPO (Direct Preference Optimization)

In 2023, researchers showed you can skip the separate reward model entirely. Instead, you build a dataset of triples: `(Prompt, Chosen Answer, Rejected Answer)`.

The idea:
1. Feed `Prompt + Chosen Answer` to the model and compute the **sequence log-probability**: the sum of the log-probabilities of each completion token. Call this `policy_chosen_logps`.
2. Do the same for `Prompt + Rejected Answer` to get `policy_rejected_logps`.
3. Compute the same two quantities under a **frozen reference model** (a copy of the model before alignment).
4. Update the active model so the chosen answer becomes *relatively* more likely than the rejected one, compared to the reference.

Two things worth pinning down:

- We work with **log-probabilities**, not raw logits. For a completion, the log-prob is the sum over its tokens of `log P(token | everything before it)`. (Logits are the unnormalized scores *before* softmax; a log-prob is what you get *after* softmax and a log. Don't conflate them.)
- The **frozen reference term is the anchor.** Without it, the model could make the chosen answer "win" by simply collapsing onto a few easy phrases or drifting away from everything it learned in pretraining (a form of reward hacking). By measuring improvement *relative to* the reference, DPO keeps the model tethered to its pretrained knowledge.

Let's simulate the core logic!

In [ ]:
import torch
import torch.nn.functional as F
torch.manual_seed(0)

def dpo_loss_simulation(policy_chosen_logps, policy_rejected_logps,
                        reference_chosen_logps, reference_rejected_logps, beta=1.0):
    """
    A simplified PyTorch representation of the DPO math.

    Each *_logps is a SEQUENCE log-probability: the sum of the log-probs of the
    completion's tokens (NOT a single logit).
    beta: how strongly we trust the preference signal vs. staying near the reference.
    """
    # 1. How much does our ACTIVE (policy) model prefer the winner vs the loser?
    policy_diff = policy_chosen_logps - policy_rejected_logps

    # 2. How much did the frozen REFERENCE model prefer the winner vs the loser?
    reference_diff = reference_chosen_logps - reference_rejected_logps

    # 3. The relative improvement over the reference (this is what anchors training).
    ratios = policy_diff - reference_diff

    # 4. The DPO loss: -log(sigmoid(beta * ratio)).
    # Low loss when the policy prefers the chosen answer MORE than the reference did.
    loss = -F.logsigmoid(beta * ratios)
    return loss

# All values below are SEQUENCE log-probabilities (sums of per-token log-probs).
beta = 1.0

# Reference (frozen base) model is roughly indifferent between the two answers.
ref_chosen   = torch.tensor([-2.2])
ref_rejected = torch.tensor([-2.1])

print("--- DPO SIMULATION (beta = 1.0) ---\n")

# CASE A (aligned): active model prefers the CHOSEN answer much more than the reference did.
act_chosen_aligned   = torch.tensor([-1.2])
act_rejected_aligned = torch.tensor([-3.4])
loss_aligned = dpo_loss_simulation(act_chosen_aligned, act_rejected_aligned,
                                   ref_chosen, ref_rejected, beta=beta)

# CASE B (misaligned): active model accidentally prefers the REJECTED answer.
act_chosen_misaligned   = torch.tensor([-3.4])
act_rejected_misaligned = torch.tensor([-1.2])
loss_misaligned = dpo_loss_simulation(act_chosen_misaligned, act_rejected_misaligned,
                                      ref_chosen, ref_rejected, beta=beta)

print(f"Aligned   (policy prefers chosen):   loss = {loss_aligned.item():.4f}")
print(f"Misaligned(policy prefers rejected): loss = {loss_misaligned.item():.4f}")
print(f"\nReference point: -log(sigmoid(0)) = ln(2) = {torch.log(torch.tensor(2.0)).item():.4f}")
print("\n-> The aligned case has a clearly LOWER loss than the misaligned case.")
print("-> Training pushes the model toward the low-loss (aligned) behavior.")

## Summary

DPO turns a preference dataset of `(prompt, chosen, rejected)` into a single, supervised-style loss. It nudges the model to prefer chosen answers over rejected ones, while the frozen reference model keeps it from drifting away from its pretrained knowledge. No separate reward model, no reinforcement-learning loop.

So far we've covered how models are built, trained, and aligned. Next we shift into **applied LLM engineering**: how to do all of this on real hardware without hundreds of gigabytes of VRAM. We start with **Module 7.1: KV Caching**, the technique that makes generation fast enough to serve.

### 🏋️ Try it yourself

1. **Sweep beta.** Loop `beta` over `[0.1, 0.5, 1.0, 5.0]` for the aligned case and print the loss for each. How does a larger `beta` change how sharply DPO reacts to the same preference gap?
2. **Build a sequence log-prob.** Given a fake tensor of per-token log-probs for a 5-token completion (e.g. `torch.tensor([-0.2, -1.1, -0.5, -0.3, -0.9])`), compute the sequence log-probability by summing them. Feed two such sums (chosen and rejected) into `dpo_loss_simulation` to see the loss end-to-end.

In [ ]:
# Your code here!
# Hint for task 1:
# for b in [0.1, 0.5, 1.0, 5.0]:
#     l = dpo_loss_simulation(act_chosen_aligned, act_rejected_aligned, ref_chosen, ref_rejected, beta=b)
#     print(f"beta={b}: loss={l.item():.4f}")